In [4]:
import numpy as np
import joblib
import optuna
import xgboost as xgb
import lightgbm as lgb
import json
from sklearn.model_selection import TimeSeriesSplit

from retail_iq.preprocessing import (
    load_raw_data,
    preprocess_dates,
    merge_datasets,
    detect_outliers_iqr,
    strict_temporal_holdout_split,
)
from retail_iq.features import FastFeatureEngineer
from retail_iq.evaluation import evaluate_model
from retail_iq.perf_utils import (
    benchmark_cache_load,
    load_or_build_feature_cache,
    optimize_dtypes_zero_copy,
)
from retail_iq.config import MODEL_DIR, PROCESSED_DATA_DIR, set_global_seed


def run_advanced_pipeline():
    set_global_seed(42)
    print("Loading data...")
    train, test, stores, oil, holidays, tx = load_raw_data()
    train, test, oil, holidays, tx = preprocess_dates([train, test, oil, holidays, tx])

    print("Merging data...")
    df = merge_datasets(train, stores, oil, holidays, tx)

    print("Engineering features...")
    cache_path = PROCESSED_DATA_DIR / "features_engineered_v3.parquet"

    def _build_features():
        fe = FastFeatureEngineer(df, transactions=tx, oil_price=oil, holidays=holidays, store_meta=stores)
        fe.add_temporal_features()\
          .add_lag_and_rolling()\
          .add_onpromotion_features()\
          .add_macroeconomic_features()\
          .add_transaction_features()\
          .add_store_metadata()\
          .add_cannibalization_features()

        out = fe.transform()
        out = out.drop(columns=["sales_lag_365d"], errors="ignore")
        out = out.dropna(subset=["sales_lag_14d"])
        out = detect_outliers_iqr(out)
        return optimize_dtypes_zero_copy(out, exclude_cols=["date"])

    features_df, from_cache = load_or_build_feature_cache(cache_path, _build_features, use_mmap=True)
    print(f"Feature cache: {'hit' if from_cache else 'miss'} -> {cache_path}")
    print("Cache load benchmark:", benchmark_cache_load(cache_path, repeats=3, use_mmap=True))

    print("Splitting data with strict 15-day temporal holdout...")
    train_df, test_df = strict_temporal_holdout_split(features_df, holdout_days=30)
    print(f"train dataframe shape: {train_df.shape}")
    print(f"test dataframe shape: {test_df.shape}")

    if len(train_df) < 200:
        raise ValueError(f"Insufficient train rows for robust tuning: {len(train_df)}")

    exclude_cols = ["id", "date", "sales", "is_outlier"]
    feature_cols = [c for c in train_df.columns if c not in exclude_cols]

    train_df = train_df.fillna(0)
    test_df = test_df.fillna(0)

    X_train = train_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
    y_train = train_df["sales"].to_numpy(dtype=np.float32, copy=False)

    X_test = test_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
    y_test = test_df["sales"].to_numpy(dtype=np.float32, copy=False)

    def objective_xgb(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 200),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "random_state": 42,
            "n_jobs": -1,
        }

        tscv = TimeSeriesSplit(n_splits=3)
        scores = []
        for train_idx, val_idx in tscv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            model = xgb.XGBRegressor(**params)
            model.fit(X_tr, np.log1p(y_tr), verbose=False)
            preds = np.expm1(model.predict(X_val))

            from sklearn.metrics import mean_squared_error

            rmse = np.sqrt(mean_squared_error(y_val, preds))
            scores.append(rmse)
        return np.mean(scores)

    print("Tuning XGBoost...")
    study_xgb = optuna.create_study(direction="minimize")
    study_xgb.optimize(objective_xgb, n_trials=2)
    best_params_xgb = study_xgb.best_params
    best_params_xgb["random_state"] = 42

    with open(MODEL_DIR / "best_params_xgb.json", "w") as f:
        json.dump(best_params_xgb, f)

    best_xgb = xgb.XGBRegressor(**best_params_xgb)
    best_xgb.fit(X_train, np.log1p(y_train))
    xgb_preds = np.expm1(best_xgb.predict(X_test))
    evaluate_model(y_test, xgb_preds, "XGBoost Tuned")
    joblib.dump(best_xgb, MODEL_DIR / "xgb_tuned_v1.pkl")

    def objective_lgb(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 200),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 20),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
            "random_state": 42,
            "verbose": -1,
            "n_jobs": -1,
        }

        tscv = TimeSeriesSplit(n_splits=3)
        scores = []
        for train_idx, val_idx in tscv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            model = lgb.LGBMRegressor(**params)
            model.fit(X_tr, np.log1p(y_tr))
            preds = np.expm1(model.predict(X_val))

            from sklearn.metrics import mean_squared_error

            rmse = np.sqrt(mean_squared_error(y_val, preds))
            scores.append(rmse)
        return np.mean(scores)

    print("Tuning LightGBM...")
    study_lgb = optuna.create_study(direction="minimize")
    study_lgb.optimize(objective_lgb, n_trials=2)
    best_params_lgb = study_lgb.best_params
    best_params_lgb["random_state"] = 42

    with open(MODEL_DIR / "best_params_lgb.json", "w") as f:
        json.dump(best_params_lgb, f)

    best_lgb = lgb.LGBMRegressor(**best_params_lgb)
    best_lgb.fit(X_train, np.log1p(y_train))
    lgb_preds = np.expm1(best_lgb.predict(X_test))
    evaluate_model(y_test, lgb_preds, "LightGBM Tuned")
    joblib.dump(best_lgb, MODEL_DIR / "lgb_tuned_v1.pkl")

    print("Advanced Modeling complete.")


if __name__ == "__main__":
    run_advanced_pipeline()


Loading data...
Merging data...
Engineering features...
Feature cache: hit -> C:\Users\mypc\Downloads\Retail_Demand_Forecasting\data\processed\features_engineered_v3.parquet
Cache load benchmark: {'load_median_s': 0.41359940002439544, 'load_min_s': 0.3823965000337921, 'load_max_s': 0.5973919999669306, 'repeats': 3.0, 'file_size_mb': 122.30453586578369}
Splitting data with strict 15-day temporal holdout...
train dataframe shape: (2922480, 42)
test dataframe shape: (53460, 42)


[I 2026-04-25 03:21:47,577] A new study created in memory with name: no-name-0fd78886-5439-462a-a209-0ea0bf9e48c9


Tuning XGBoost...


[I 2026-04-25 03:23:26,527] Trial 0 finished with value: 283.4942694025151 and parameters: {'n_estimators': 189, 'max_depth': 8, 'learning_rate': 0.1595388057293031, 'subsample': 0.7192619201298629, 'colsample_bytree': 0.6293107748876356}. Best is trial 0 with value: 283.4942694025151.
[I 2026-04-25 03:24:10,036] Trial 1 finished with value: 305.0855877333123 and parameters: {'n_estimators': 58, 'max_depth': 10, 'learning_rate': 0.15899446950990723, 'subsample': 0.6985120284150408, 'colsample_bytree': 0.6415782342211556}. Best is trial 0 with value: 283.4942694025151.
[I 2026-04-25 03:25:09,445] A new study created in memory with name: no-name-1a5db252-452a-44a4-be5c-134dc4c846f8


XGBoost Tuned: RMSLE=0.3753, RMSE=200.78, MAPE=30.87%, R²=0.9747
Tuning LightGBM...


c:\Users\mypc\Downloads\Retail_Demand_Forecasting\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\mypc\Downloads\Retail_Demand_Forecasting\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\mypc\Downloads\Retail_Demand_Forecasting\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-04-25 03:26:13,744] Trial 0 finished with value: 366.08991204468947 and parameters: {'n_estimators': 175, 'max_depth': 5, 'learning_rate': 0.026399831437044878, 'num_leaves': 73, 'min_child_samples': 8, 'feature_fraction': 0.8479823097503539}. Best is trial 0 with value: 366.08991204468947.
c:\Users\mypc\Downloads\Retail_

LightGBM Tuned: RMSLE=0.3889, RMSE=212.65, MAPE=32.52%, R²=0.9716
Advanced Modeling complete.
